==================================================
                 TASK SUBMISSION
==================================================

Full Name:         Pranav Rase
Internship Track:  Data Analytics
Task Title:        Task 3 - Cleaning Data (Titanic Dataset)

==================================================

In [4]:
import pandas as pd
import numpy as np
file_path = 'Titanic-Dataset.csv'
df_raw = pd.read_csv(file_path)

print("="*80)
print("                    INITIAL DATA QUALITY REPORT")
print("="*80)
print(f"Total Rows: {df_raw.shape[0]} | Total Columns: {df_raw.shape[1]}")
print(f"Duplicate Rows Count: {df_raw.duplicated().sum()}")
print("\n--- Missing Values per Column ---")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

print("\n--- Current Column Data Types ---")
print(df_raw.dtypes)

print("\n--- Raw Numeric Summary Statistics ---")
print(df_raw.describe().T[['mean', 'std', 'min', '50%', 'max']])

# Create copy for cleaning pipeline
df_clean = df_raw.copy()

                    INITIAL DATA QUALITY REPORT
Total Rows: 891 | Total Columns: 12
Duplicate Rows Count: 0

--- Missing Values per Column ---
Age         177
Cabin       687
Embarked      2
dtype: int64

--- Current Column Data Types ---
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

--- Raw Numeric Summary Statistics ---
                   mean         std   min       50%       max
PassengerId  446.000000  257.353842  1.00  446.0000  891.0000
Survived       0.383838    0.486592  0.00    0.0000    1.0000
Pclass         2.308642    0.836071  1.00    3.0000    3.0000
Age           29.699118   14.526497  0.42   28.0000   80.0000
SibSp          0.523008    1.102743  0.00    0.0000    8.0000
Parch          0.381594    0.806057  0.00    0.0000    6.00

In [5]:
# Age: Group-wise median imputation based on (Pclass, Sex)
df_clean['Age'] = df_clean['Age'].fillna(
    df_clean.groupby(['Pclass', 'Sex'])['Age'].transform('median')
)

# Embarked: Mode imputation ('S')
mode_embarked = df_clean['Embarked'].mode()[0]
df_clean['Embarked'] = df_clean['Embarked'].fillna(mode_embarked)

# Cabin: Drop column due to excessive missingness (>77%)
df_clean.drop(columns=['Cabin'], inplace=True)

In [6]:
num_duplicates = df_clean.duplicated().sum()
if num_duplicates > 0:
    df_clean.drop_duplicates(inplace=True)
print(f"[Duplicates]: Removed {num_duplicates} duplicate rows.")

[Duplicates]: Removed 0 duplicate rows.


In [7]:
# Standardise string formatting (trim whitespaces, capitalize/uppercase)
df_clean['Sex'] = df_clean['Sex'].str.strip().str.capitalize()
df_clean['Embarked'] = df_clean['Embarked'].str.strip().str.upper()

In [8]:
# Calculate IQR boundaries
Q1 = df_clean['Fare'].quantile(0.25)
Q3 = df_clean['Fare'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + (1.5 * IQR)

print(f"[Outliers]: Fare IQR Upper Limit calculated as ${upper_bound:.2f}")
outlier_count = (df_clean['Fare'] > upper_bound).sum()
print(f"[Outliers]: Capping {outlier_count} Fare outliers at upper limit (${upper_bound:.2f}).")

# Cap outliers exceeding upper bound
df_clean['Fare'] = np.where(df_clean['Fare'] > upper_bound, upper_bound, df_clean['Fare'])

[Outliers]: Fare IQR Upper Limit calculated as $65.63
[Outliers]: Capping 116 Fare outliers at upper limit ($65.63).


In [9]:
df_clean['PassengerId'] = df_clean['PassengerId'].astype(str)
df_clean['Survived'] = df_clean['Survived'].astype('category')
df_clean['Pclass'] = df_clean['Pclass'].astype('category')
df_clean['Sex'] = df_clean['Sex'].astype('category')
df_clean['Embarked'] = df_clean['Embarked'].astype('category')

In [10]:
output_filename = 'Titanic-Dataset-Cleaned.csv'
df_clean.to_csv(output_filename, index=False)
print(f"Successfully exported cleaned dataset to '{output_filename}'.")

Successfully exported cleaned dataset to 'Titanic-Dataset-Cleaned.csv'.


In [11]:
summary_table = pd.DataFrame({
    'Metric / Column': [
        'Total Row Count', 
        'PassengerId Dtype', 
        'Survived Dtype', 
        'Age Missing Count', 
        'Embarked Missing Count', 
        'Cabin Missing Count', 
        'Max Fare Value'
    ],
    'Before Cleaning': [
        len(df_raw), 
        str(df_raw['PassengerId'].dtype), 
        str(df_raw['Survived'].dtype), 
        df_raw['Age'].isnull().sum(), 
        df_raw['Embarked'].isnull().sum(), 
        f"{df_raw['Cabin'].isnull().sum()} (Column present)", 
        f"${df_raw['Fare'].max():.2f}"
    ],
    'After Cleaning': [
        len(df_clean), 
        str(df_clean['PassengerId'].dtype), 
        str(df_clean['Survived'].dtype), 
        df_clean['Age'].isnull().sum(), 
        df_clean['Embarked'].isnull().sum(), 
        "Column Dropped", 
        f"${df_clean['Fare'].max():.2f}"
    ]
})

print("="*80)
print("                    BEFORE VS. AFTER CLEANING SUMMARY")
print("="*80)
print(summary_table.to_string(index=False))

                    BEFORE VS. AFTER CLEANING SUMMARY
       Metric / Column      Before Cleaning After Cleaning
       Total Row Count                  891            891
     PassengerId Dtype                int64            str
        Survived Dtype                int64       category
     Age Missing Count                  177              0
Embarked Missing Count                    2              0
   Cabin Missing Count 687 (Column present) Column Dropped
        Max Fare Value              $512.33         $65.63
